# Calculating and pickling results of simulations

**Objective -** To simulate a population of short GRBs thanks to the saved configurations, calculate some results from these simulations and save these results in another Pickle file.

- For each configuration of a `all_dico_<thetaCore>.pkl` file, for each type of jet and several results are calculated : $t_{obs}$, $mag_{min}$, the axis, the observability and the flux in photons in the $\gamma$ range. All these results are put in a dictionary along with the configuration.
- 1000 configurations $ \Longrightarrow $ list of the 1000 dictionaries of results,
- The list is saved in a pickle.

**Output :** `all_dico_results_<jettype>_<theta_c>.pkl` where `<jettype>` = `TH` (Top-Hat), `G` (Gaussian) or `PL` (Power-Law), and `<theta_c>` = `005` (0.05 radians) or `015` (0.15 radians).

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import afterglowpy as grb
import math 
import pandas as pd
from astropy.cosmology import Planck18 as cosmo
from scipy.integrate import quad

In [2]:
def ObsTime(t, mag):
    index = []
    for i in range(len(mag)-1):
        if mag[i] < 24.5:
            index.append(i)
    if not index:
        return 0
    else:
        dt = (t[max(index)] - t[min(index)])*grb.sec2day
        return dt

## 1. Top-Hat Jet

In [3]:
#file = open('../data/simulations/all_dico_results_TH_005.pkl', 'wb')
file = open('../data/simulations/all_dico_results_TH_015.pkl', 'wb')

file_open = open('../data/simulations/all_dico_015.pkl', 'rb')
configs_open = pickle.load(file_open)


Z_results = {'jetType' :      -1,          # Jet Type
             'config' :       {},          # Dictionary Z
             't_obs' :        0,           # Observability duration in days
             'mag_min' :      0,           # Minimum magnitude of the afterglow
             'axis' :         'on',        # On-axis or Off-axis
             'observable' :   'no',        # Observability of the afterglow : no or yes for >4. or >7. in days
             'F_gamma' :      '0'}         # Flux of photons above 30 MeV in ph/cm2/s

all_Z_results = []
dt = []


for i in range(len(configs_open)):
    
    t = np.geomspace(1.0e2, 1.0e9, 300)
    nu = 5.0e14
    
    Z = configs_open[i]
    Z['jetType'] = grb.jet.TopHat
    Z['thetaCore'] =  0.15
    
    mag = -2.5 * np.log10(grb.fluxDensity(t, nu, **Z)*1.0e-26) - 48.6
    dt.append(ObsTime(t, mag))
    
    # Filling of Z_results dictionary
    if Z['jetType'] == grb.jet.TopHat:
        Z_results['jetType'] = 'TopHat'
    elif Z['jetType'] == grb.jet.Gaussian:
        Z_results['jetType'] = 'Gaussian'
    elif Z['jetType'] == grb.jet.PowerLaw:
        Z_results['jetType'] = 'PowerLaw'
        
    Z_results['config'] = Z
    Z_results['t_obs'] = dt[i]
    Z_results['mag_min'] = min(mag)
    
    if Z['thetaCore'] < Z['thetaObs']:
        Z_results['axis'] = 'off'
    else:
        Z_results['axis'] = 'on'
    
    if dt[i] < 4.:
        Z_results['observable'] = 'no'
    elif dt[i] >= 4.:
        Z_results['observable'] = '> 4 days'
        if dt[i] >= 7.:
            Z_results['observable'] = '> 7 days'
    
    imax = np.where(mag == min(mag))
    t_max = t[imax[0]]

    if len(t_max) == 1:
        
        # Range of LAT
        nua = 1.0e8 * 1.6e-19 / 6.62e-34    # 10^5 keV
        nub = 5.0e10 * 1.6e-19 / 6.62e-34    # 5 x 10^ keV

        nu = np.geomspace(nua, nub, num=100)
        Fnu_max = grb.fluxDensity(t_max, nu, **Z)   # Flux en mJy

        delta_nu = nub - nua

        Nnu = []     # Flux calculated with wavelength
        for i in range(len(nu)):
            Nnu.append(Fnu_max[i] * 10**-29 / (nu[i] * 6.62e-34))

        Z_results['F_gamma'] = sum(Nnu) * delta_nu / 10**4 # Flux in photon/cm2/s
        
    else:
        Z_results['F_gamma'] = 0
     
    all_Z_results.append(Z_results.copy())

    
pickle.dump(all_Z_results, file)
     
file_open.close()
file.close()

## 2. Gaussian Jet
#### 2.1 $\theta_c$ = 0.05 radians

In [4]:
file = open('../data/simulations/all_dico_results_G_005.pkl', 'wb')

file_open = open('../data/simulations/all_dico_005.pkl', 'rb')
configs_open = pickle.load(file_open)

Z_results = {'jetType' :      -1,          # Jet Type
             'config' :       {},          # Dictionary Z
             't_obs' :        0,           # Observability duration in days
             'mag_min' :      0,           # Minimum magnitude of the afterglow
             'axis' :         'on',        # On-axis or Off-axis
             'observable' :   'no',        # Observability of the afterglow : no or yes for >4. or >7. in days
             'F_gamma' :      '0'}         # Flux of photons above 100 keV in ph/cm2/s


all_Z_results = []
dt = []


for i in range(len(configs_open)):
    
    t = np.geomspace(1.0e2, 1.0e9, 300)
    nu = 5.0e14
    
    Z = configs_open[i]
    Z['jetType'] = grb.jet.Gaussian
    Z['thetaCore'] = 0.05

    mag = -2.5 * np.log10(grb.fluxDensity(t, nu, **Z)*1.0e-26) - 48.6
    dt.append(ObsTime(t, mag))
    
    # Filling of Z_results dictionary
    if Z['jetType'] == grb.jet.TopHat:
        Z_results['jetType'] = 'TopHat'
    elif Z['jetType'] == grb.jet.Gaussian:
        Z_results['jetType'] = 'Gaussian'
    elif Z['jetType'] == grb.jet.PowerLaw:
        Z_results['jetType'] = 'PowerLaw'
        
    Z_results['config'] = Z
    Z_results['t_obs'] = dt[i]
    Z_results['mag_min'] = min(mag)
    
    if Z['thetaWing'] < Z['thetaObs']:
        Z_results['axis'] = 'off'
    else:
        Z_results['axis'] = 'on'
    
    if dt[i] < 4.:
        Z_results['observable'] = 'no'
    elif dt[i] >= 4.:
        Z_results['observable'] = '> 4 days'
        if dt[i] >= 7.:
            Z_results['observable'] = '> 7 days'
            
    imax = np.where(mag == min(mag))
    t_max = t[imax]

    if len(t_max) == 1:
        # Range of LAT
        nua = 1.0e8 * 1.6e-19 / 6.62e-34    # 10^5 keV
        nub = 5.0e10 * 1.6e-19 / 6.62e-34    # 5 x 10^7 keV


        nu = np.geomspace(nua, nub, num=100)
        Fnu_max = grb.fluxDensity(t_max, nu, **Z)   # Flux en mJy

        delta_nu = nub - nua

        Nnu = []     # Flux calculated with wavelength
        for i in range(len(nu)):
            Nnu.append(Fnu_max[i] * 10**-29 / (nu[i] * 6.62e-34))

        Z_results['F_gamma'] = sum(Nnu) * delta_nu / 10**4 # Flux in photon/cm2/s
        
    else:
        Z_results['F_gamma'] = 0
    
    all_Z_results.append(Z_results.copy())
    

pickle.dump(all_Z_results, file)
     
file_open.close()
file.close()

#### 2.2 $\theta_c$ = 0.15 radians

In [5]:
file = open('../data/simulations/all_dico_results_G_015.pkl', 'wb')

file_open = open('../data/simulations/all_dico_015.pkl', 'rb')
configs_open = pickle.load(file_open)

Z_results = {'jetType' :      -1,          # Jet Type
             'config' :       {},          # Dictionary Z
             't_obs' :        0,           # Observability duration in days
             'mag_min' :      0,           # Minimum magnitude of the afterglow
             'axis' :         'on',        # On-axis or Off-axis
             'observable' :   'no',        # Observability of the afterglow : no or yes for >4. or >7. in days
             'F_gamma' :      '0'}         # Flux of photons above 100 keV in ph/cm2/s


all_Z_results = []
dt = []


for i in range(len(configs_open)):
    
    t = np.geomspace(1.0e2, 1.0e9, 300)
    nu = 5.0e14
    
    Z = configs_open[i]
    Z['jetType'] = grb.jet.Gaussian
    Z['thetaCore'] = 0.15

    mag = -2.5 * np.log10(grb.fluxDensity(t, nu, **Z)*1.0e-26) - 48.6
    dt.append(ObsTime(t, mag))
    
    # Filling of Z_results dictionary
    if Z['jetType'] == grb.jet.TopHat:
        Z_results['jetType'] = 'TopHat'
    elif Z['jetType'] == grb.jet.Gaussian:
        Z_results['jetType'] = 'Gaussian'
    elif Z['jetType'] == grb.jet.PowerLaw:
        Z_results['jetType'] = 'PowerLaw'
        
    Z_results['config'] = Z
    Z_results['t_obs'] = dt[i]
    Z_results['mag_min'] = min(mag)
    
    if Z['thetaWing'] < Z['thetaObs']:
        Z_results['axis'] = 'off'
    else:
        Z_results['axis'] = 'on'
    
    if dt[i] < 4.:
        Z_results['observable'] = 'no'
    elif dt[i] >= 4.:
        Z_results['observable'] = '> 4 days'
        if dt[i] >= 7.:
            Z_results['observable'] = '> 7 days'
            
    imax = np.where(mag == min(mag))    
    t_max = t[imax]
    
    if len(t_max) == 1:
        # Range of LAT
        nua = 1.0e8 * 1.6e-19 / 6.62e-34    # 10^5 keV
        nub = 5.0e10 * 1.6e-19 / 6.62e-34    # 5 x 10^7 keV


        nu = np.geomspace(nua, nub, num=100)
        Fnu_max = grb.fluxDensity(t_max, nu, **Z)   # Flux en mJy

        delta_nu = nub - nua

        Nnu = []     # Flux calculated with wavelength
        for i in range(len(nu)):
            Nnu.append(Fnu_max[i] * 10**-29 / (nu[i] * 6.62e-34))

        Z_results['F_gamma'] = sum(Nnu) * delta_nu / 10**4 # Flux in photon/cm2/s
        
    else:
        Z_results['F_gamma'] = 0
    
    all_Z_results.append(Z_results.copy())
    

pickle.dump(all_Z_results, file)
     
file_open.close()
file.close()

## 3. Power-Law Jet
#### 3.1 $\theta_c$ = 0.05 radians

In [6]:
file = open('../data/simulations/all_dico_results_PL_005.pkl', 'wb')

file_open = open('../data/simulations/all_dico_005.pkl', 'rb')
configs_open = pickle.load(file_open)

Z_results = {'jetType' :      -1,          # Jet Type
             'config' :       {},          # Dictionary Z
             't_obs' :        0,           # Observability duration in days
             'mag_min' :      0,           # Minimum magnitude of the afterglow
             'axis' :         'on',        # On-axis or Off-axis
             'observable' :   'no',        # Observability of the afterglow : no or yes for >4. or >7. in days
             'F_gamma' :      '0'}         # Flux of photons above 100 keV in ph/cm2/s


all_Z_results = []
dt = []


for i in range(len(configs_open)):
    
    t = np.geomspace(1.0e2, 1.0e9, 300)
    nu = 5.0e14
    
    Z = configs_open[i]
    Z['jetType'] = grb.jet.PowerLaw
    Z['thetaCore'] = 0.05

    mag = -2.5 * np.log10(grb.fluxDensity(t, nu, **Z)*1.0e-26) - 48.6
    dt.append(ObsTime(t, mag))
    
    # Filling of Z_results dictionary
    if Z['jetType'] == grb.jet.TopHat:
        Z_results['jetType'] = 'TopHat'
    elif Z['jetType'] == grb.jet.Gaussian:
        Z_results['jetType'] = 'Gaussian'
    elif Z['jetType'] == grb.jet.PowerLaw:
        Z_results['jetType'] = 'PowerLaw'
        
    Z_results['config'] = Z
    Z_results['t_obs'] = dt[i]
    Z_results['mag_min'] = min(mag)
    
    if Z['thetaWing'] < Z['thetaObs']:
        Z_results['axis'] = 'off'
    else:
        Z_results['axis'] = 'on'
    
    if dt[i] < 4.:
        Z_results['observable'] = 'no'
    elif dt[i] >= 4.:
        Z_results['observable'] = '> 4 days'
        if dt[i] >= 7.:
            Z_results['observable'] = '> 7 days'
            
    imax = np.where(mag == min(mag))
    t_max = t[imax]

    if len(t_max) == 1:
        # Range of LAT
        nua = 1.0e8 * 1.6e-19 / 6.62e-34    # 10^5 keV
        nub = 5.0e10 * 1.6e-19 / 6.62e-34    # 5 x 10^7 keV


        nu = np.geomspace(nua, nub, num=100)
        Fnu_max = grb.fluxDensity(t_max, nu, **Z)   # Flux en mJy

        delta_nu = nub - nua

        Nnu = []     # Flux calculated with wavelength
        for i in range(len(nu)):
            Nnu.append(Fnu_max[i] * 10**-29 / (nu[i] * 6.62e-34))

        Z_results['F_gamma'] = sum(Nnu)  * delta_nu / 10**4 # Flux in photon/cm2/s
        
    else:
        Z_results['F_gamma'] = 0
    
    all_Z_results.append(Z_results.copy())
    

pickle.dump(all_Z_results, file)
     
file_open.close()
file.close()

#### 3.2 $\theta_c$ = 0.15 radians

In [7]:
file = open('../data/simulations/all_dico_results_PL_015.pkl', 'wb')

file_open = open('../data/simulations/all_dico_015.pkl', 'rb')
configs_open = pickle.load(file_open)

Z_results = {'jetType' :      -1,          # Jet Type
             'config' :       {},          # Dictionary Z
             't_obs' :        0,           # Observability duration in days
             'mag_min' :      0,           # Minimum magnitude of the afterglow
             'axis' :         'on',        # On-axis or Off-axis
             'observable' :   'no',        # Observability of the afterglow : no or yes for >4. or >7. in days
             'F_gamma' :      '0'}         # Flux of photons above 100 keV in ph/cm2/s


all_Z_results = []
dt = []


for i in range(len(configs_open)):
    
    t = np.geomspace(1.0e2, 1.0e9, 300)
    nu = 5.0e14
    
    Z = configs_open[i]
    Z['jetType'] = grb.jet.PowerLaw
    Z['thetaCore'] = 0.15

    mag = -2.5 * np.log10(grb.fluxDensity(t, nu, **Z)*1.0e-26) - 48.6
    dt.append(ObsTime(t, mag))
    
    # Filling of Z_results dictionary
    if Z['jetType'] == grb.jet.TopHat:
        Z_results['jetType'] = 'TopHat'
    elif Z['jetType'] == grb.jet.Gaussian:
        Z_results['jetType'] = 'Gaussian'
    elif Z['jetType'] == grb.jet.PowerLaw:
        Z_results['jetType'] = 'PowerLaw'
        
    Z_results['config'] = Z
    Z_results['t_obs'] = dt[i]
    Z_results['mag_min'] = min(mag)
    
    if Z['thetaWing'] < Z['thetaObs']:
        Z_results['axis'] = 'off'
    else:
        Z_results['axis'] = 'on'
    
    if dt[i] < 4.:
        Z_results['observable'] = 'no'
    elif dt[i] >= 4.:
        Z_results['observable'] = '> 4 days'
        if dt[i] >= 7.:
            Z_results['observable'] = '> 7 days'
            
    imax = np.where(mag == min(mag))
    t_max = t[imax]

    if len(t_max) == 1:
        # Range of LAT
        nua = 1.0e8 * 1.6e-19 / 6.62e-34    # 10^5 keV
        nub = 5.0e10 * 1.6e-19 / 6.62e-34    # 5 x 10^7 keV

        nu = np.geomspace(nua, nub, num=100)
        Fnu_max = grb.fluxDensity(t_max, nu, **Z)   # Flux en mJy

        delta_nu = nub - nua

        Nnu = []     # Flux calculated with wavelength
        for i in range(len(nu)):
            Nnu.append(Fnu_max[i] * 10**-29 / (nu[i] * 6.62e-34))

        Z_results['F_gamma'] = sum(Nnu)  * delta_nu / 10**4 # Flux in photon/cm2/s
        
    else:
        Z_results['F_gamma'] = 0
    
    all_Z_results.append(Z_results.copy())
    

pickle.dump(all_Z_results, file)
     
file_open.close()
file.close()